In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import matplotlib.pyplot as plt
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("switch colab runtime to GPU first")
device = "cuda"

N_TRAIN = 1000
N_TEST = 200
M_SENSORS = 40
P_POINTS = 100
GRF_LENGTH_SCALE = 0.2

WANG_ITERS = 80000
WANG_LR = 1e-3
WANG_LR_DECAY_STEPS = 5000
WANG_LR_DECAY_RATE = 0.9
WANG_WIDTH = 50
WANG_DEPTH = 5

LAMBDA_PHYS = 1.0
LAMBDA_IC = 20.0
LAMBDA_BC = 20.0
BATCH = 5000
N_PINN_COLLOC = 1000

NX = 100
CFL_SAFETY = 0.4

RUN_TAG = "advection_ablation"

def sample_grf(n_samples, n_points, length_scale):
    x = np.linspace(0, 1, n_points)
    xi, xj = np.meshgrid(x, x)
    K = np.exp(-0.5 * ((xi - xj) / length_scale) ** 2) + 1e-10 * np.eye(n_points)
    L = np.linalg.cholesky(K)
    z = np.random.randn(n_samples, n_points)
    return z @ L.T

def solve_advection_upwind(u_at_sensors, nx=NX):
    x = np.linspace(0, 1, nx)
    dx = x[1] - x[0]
    u_x = np.interp(x, np.linspace(0, 1, len(u_at_sensors)), u_at_sensors)
    u_x = np.clip(u_x, 0.5, 2.0)
    u_max = u_x.max()
    dt = CFL_SAFETY * dx / u_max
    nt = int(np.ceil(1.0 / dt)) + 1
    dt = 1.0 / (nt - 1)
    s = np.zeros((nt, nx), dtype=np.float32)
    s[0] = np.sin(np.pi * x)
    for n in range(nt - 1):
        t_next = (n + 1) * dt
        s[n + 1, 1:] = s[n, 1:] - u_x[1:] * dt / dx * (s[n, 1:] - s[n, :-1])
        s[n + 1, 0] = np.sin(np.pi / 2 * t_next)
        if not np.isfinite(s[n + 1]).all():
            raise RuntimeError(f"solver blew up at step {n+1}, u_max={u_max:.3f}")
    t = np.linspace(0, 1, nt)
    nt_out = 100
    t_out = np.linspace(0, 1, nt_out)
    s_out = np.zeros((nt_out, nx), dtype=np.float32)
    for i in range(nx):
        s_out[:, i] = np.interp(t_out, t, s[:, i])
    return x, t_out, s_out

x_sensors = np.linspace(0, 1, M_SENSORS)

def make_dataset(n_samples):
    u_raw = sample_grf(n_samples, M_SENSORS, GRF_LENGTH_SCALE).astype(np.float32)
    u_at_sensors = (u_raw - u_raw.min(axis=1, keepdims=True) + 1.0).astype(np.float32)
    u_at_sensors = np.clip(u_at_sensors, 0.5, 2.0)
    xt_query = np.random.rand(n_samples, P_POINTS, 2).astype(np.float32)
    s_query = np.zeros((n_samples, P_POINTS), dtype=np.float32)
    s_full_list = []
    for i in range(n_samples):
        x_grid, t_grid, s_grid = solve_advection_upwind(u_at_sensors[i])
        s_full_list.append(s_grid)
        for p in range(P_POINTS):
            xi_q, ti_q = xt_query[i, p]
            ix = int(np.clip(xi_q * (len(x_grid) - 1), 0, len(x_grid) - 2))
            it = int(np.clip(ti_q * (len(t_grid) - 1), 0, len(t_grid) - 2))
            s_query[i, p] = s_grid[it, ix]
    return u_at_sensors, xt_query, s_query, np.array(s_full_list)

print("generating dataset (shared across all trainings)")
u_train, xt_train, s_train, _ = make_dataset(N_TRAIN)
u_test, xt_test, s_test, s_test_full = make_dataset(N_TEST)
print(f"dataset ready | s range: [{s_test_full.min():.3f}, {s_test_full.max():.3f}]")

def mlp(sizes, act):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(torch.nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return torch.nn.Sequential(*layers)

class PINN(torch.nn.Module):
    def __init__(self, width, depth):
        super().__init__()
        self.net = mlp([2] + [width] * depth + [1], torch.nn.Tanh)
    def forward(self, xt):
        return self.net(xt)

class DeepONet(torch.nn.Module):
    def __init__(self, width, depth):
        super().__init__()
        self.branch = mlp([M_SENSORS] + [width] * depth, torch.nn.Tanh)
        self.trunk = mlp([2] + [width] * depth, torch.nn.Tanh)
        self.bias = torch.nn.Parameter(torch.zeros(1))
    def forward(self, u, xt):
        return (self.branch(u) * self.trunk(xt)).sum(dim=1, keepdim=True) + self.bias

def rel_l2(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-10)

def u_interp_np_batch(x_query_np, u_vals_np):
    idx_left = np.clip(np.searchsorted(x_sensors, x_query_np) - 1, 0, M_SENSORS - 2)
    x_left = x_sensors[idx_left]
    x_right = x_sensors[idx_left + 1]
    alpha = ((x_query_np - x_left) / (x_right - x_left)).astype(np.float32)
    if u_vals_np.ndim == 1:
        return u_vals_np[idx_left] + alpha * (u_vals_np[idx_left + 1] - u_vals_np[idx_left])
    else:
        n = len(x_query_np)
        u_left = u_vals_np[np.arange(n), idx_left]
        u_right = u_vals_np[np.arange(n), idx_left + 1]
        return u_left + alpha * (u_right - u_left)

xg = np.linspace(0, 1, NX)
tg = np.linspace(0, 1, 100)
nx, nt = len(xg), len(tg)
XX, TT = np.meshgrid(xg, tg)

print("\n" + "=" * 60)
print("PINN (trained once, Wang setup)")
print("=" * 60)
u_pinn = u_test[0]
pinn_net = PINN(WANG_WIDTH, WANG_DEPTH).to(device)
pinn_opt = torch.optim.Adam(pinn_net.parameters(), lr=WANG_LR)
pinn_sched = torch.optim.lr_scheduler.StepLR(pinn_opt, step_size=WANG_LR_DECAY_STEPS, gamma=WANG_LR_DECAY_RATE)

for step in range(WANG_ITERS + 1):
    xt_c = torch.rand(N_PINN_COLLOC, 2, device=device, requires_grad=True)
    s_pred = pinn_net(xt_c)
    grads = torch.autograd.grad(s_pred, xt_c, grad_outputs=torch.ones_like(s_pred), create_graph=True)[0]
    ds_dx = grads[:, 0:1]; ds_dt = grads[:, 1:2]
    x_np = xt_c[:, 0].detach().cpu().numpy()
    u_at_x = torch.tensor(u_interp_np_batch(x_np, u_pinn).reshape(-1, 1), device=device)
    loss_phys = ((ds_dt + u_at_x * ds_dx) ** 2).mean()
    x_ic = torch.rand(200, 1, device=device)
    xt_ic = torch.cat([x_ic, torch.zeros_like(x_ic)], dim=1)
    loss_ic = ((pinn_net(xt_ic) - torch.sin(np.pi * x_ic)) ** 2).mean()
    t_bc = torch.rand(200, 1, device=device)
    xt_bc = torch.cat([torch.zeros_like(t_bc), t_bc], dim=1)
    loss_bc = ((pinn_net(xt_bc) - torch.sin(np.pi / 2 * t_bc)) ** 2).mean()
    loss = LAMBDA_PHYS * loss_phys + LAMBDA_IC * loss_ic + LAMBDA_BC * loss_bc
    pinn_opt.zero_grad(); loss.backward(); pinn_opt.step(); pinn_sched.step()
    if step % 20000 == 0:
        print(f"pinn step {step:6d} | phys {loss_phys.item():.3e} | ic {loss_ic.item():.3e} | bc {loss_bc.item():.3e}")

with torch.no_grad():
    _, _, s_true_pinn = solve_advection_upwind(u_pinn)
    xt_eval = torch.tensor(np.stack([XX.flatten(), TT.flatten()], axis=1).astype(np.float32), device=device)
    s_pinn_pred = pinn_net(xt_eval).cpu().numpy().reshape(nt, nx)
pinn_error = rel_l2(s_pinn_pred, s_true_pinn)
print(f"PINN rel L2: {pinn_error:.4%}")

print("\n" + "=" * 60)
print("DeepONet (trained once, Wang setup)")
print("=" * 60)
don = DeepONet(WANG_WIDTH, WANG_DEPTH).to(device)
don_opt = torch.optim.Adam(don.parameters(), lr=WANG_LR)
don_sched = torch.optim.lr_scheduler.StepLR(don_opt, step_size=WANG_LR_DECAY_STEPS, gamma=WANG_LR_DECAY_RATE)

u_train_t = torch.tensor(u_train.repeat(P_POINTS, 0).reshape(N_TRAIN * P_POINTS, M_SENSORS), device=device)
xt_train_t = torch.tensor(xt_train.reshape(-1, 2), device=device)
s_train_t = torch.tensor(s_train.reshape(-1, 1), device=device)

for step in range(WANG_ITERS + 1):
    idx = torch.randint(0, u_train_t.shape[0], (BATCH,))
    pred = don(u_train_t[idx], xt_train_t[idx])
    loss = ((pred - s_train_t[idx]) ** 2).mean()
    don_opt.zero_grad(); loss.backward(); don_opt.step(); don_sched.step()
    if step % 20000 == 0:
        print(f"don step {step:6d} | data {loss.item():.3e}")

with torch.no_grad():
    u_test_t = torch.tensor(u_test, device=device)
    don_preds = np.zeros((N_TEST, nt, nx))
    for i in range(N_TEST):
        u_i = u_test_t[i:i+1].repeat(nt * nx, 1)
        xt_i = torch.tensor(np.stack([XX.flatten(), TT.flatten()], axis=1).astype(np.float32), device=device)
        don_preds[i] = don(u_i, xt_i).cpu().numpy().reshape(nt, nx)
don_errors = [rel_l2(don_preds[i], s_test_full[i]) for i in range(N_TEST)]
don_mean = np.mean(don_errors); don_std = np.std(don_errors)
print(f"DeepONet mean rel L2: {don_mean:.4%} +/- {don_std:.4%}")

def train_pi_deeponet(variant_name, use_bc, width, depth, use_lr_decay, iters):
    print(f"\n{'=' * 60}\nPI-DeepONet: {variant_name}\n{'=' * 60}")
    pinet = DeepONet(width, depth).to(device)
    pi_opt = torch.optim.Adam(pinet.parameters(), lr=WANG_LR)
    if use_lr_decay:
        pi_sched = torch.optim.lr_scheduler.StepLR(pi_opt, step_size=WANG_LR_DECAY_STEPS, gamma=WANG_LR_DECAY_RATE)
    u_all_train_t = torch.tensor(u_train, device=device)

    for step in range(iters + 1):
        idx = torch.randint(0, u_train_t.shape[0], (BATCH,))
        pred = pinet(u_train_t[idx], xt_train_t[idx])
        loss_data = ((pred - s_train_t[idx]) ** 2).mean()

        idx_p = torch.randint(0, N_TRAIN, (BATCH,))
        u_p = u_all_train_t[idx_p]
        xt_p = torch.rand(BATCH, 2, device=device, requires_grad=True)
        s_p = pinet(u_p, xt_p)
        grads = torch.autograd.grad(s_p, xt_p, grad_outputs=torch.ones_like(s_p), create_graph=True)[0]
        ds_dx = grads[:, 0:1]; ds_dt = grads[:, 1:2]
        x_np = xt_p[:, 0].detach().cpu().numpy()
        u_vals_np = u_train[idx_p.cpu().numpy()]
        u_at_x = torch.tensor(u_interp_np_batch(x_np, u_vals_np).reshape(-1, 1), device=device)
        loss_phys = ((ds_dt + u_at_x * ds_dx) ** 2).mean()

        x_ic = torch.rand(BATCH, 1, device=device)
        xt_ic = torch.cat([x_ic, torch.zeros_like(x_ic)], dim=1)
        loss_ic = ((pinet(u_all_train_t[idx_p], xt_ic) - torch.sin(np.pi * x_ic)) ** 2).mean()

        total_loss = loss_data + LAMBDA_PHYS * loss_phys + LAMBDA_IC * loss_ic

        if use_bc:
            t_bc = torch.rand(BATCH, 1, device=device)
            xt_bc = torch.cat([torch.zeros_like(t_bc), t_bc], dim=1)
            loss_bc = ((pinet(u_all_train_t[idx_p], xt_bc) - torch.sin(np.pi / 2 * t_bc)) ** 2).mean()
            total_loss = total_loss + LAMBDA_BC * loss_bc
        else:
            loss_bc = torch.tensor(0.0)

        pi_opt.zero_grad(); total_loss.backward(); pi_opt.step()
        if use_lr_decay:
            pi_sched.step()

        if step % 20000 == 0:
            print(f"  step {step:6d} | data {loss_data.item():.3e} | phys {loss_phys.item():.3e} | ic {loss_ic.item():.3e} | bc {loss_bc.item():.3e}")

    with torch.no_grad():
        pi_preds = np.zeros((N_TEST, nt, nx))
        for i in range(N_TEST):
            u_i = u_test_t[i:i+1].repeat(nt * nx, 1)
            xt_i = torch.tensor(np.stack([XX.flatten(), TT.flatten()], axis=1).astype(np.float32), device=device)
            pi_preds[i] = pinet(u_i, xt_i).cpu().numpy().reshape(nt, nx)
    errors = [rel_l2(pi_preds[i], s_test_full[i]) for i in range(N_TEST)]
    mean_err = np.mean(errors); std_err = np.std(errors)
    print(f"  {variant_name} rel L2: {mean_err:.4%} +/- {std_err:.4%}")
    return mean_err, std_err

ablation_results = {}
ablation_results["Wang baseline"] = train_pi_deeponet("Wang baseline (all on)", use_bc=True, width=WANG_WIDTH, depth=WANG_DEPTH, use_lr_decay=True, iters=WANG_ITERS)
ablation_results["A1 no BC"] = train_pi_deeponet("A1: no boundary condition", use_bc=False, width=WANG_WIDTH, depth=WANG_DEPTH, use_lr_decay=True, iters=WANG_ITERS)
ablation_results["A2 small net"] = train_pi_deeponet("A2: small network (2x40)", use_bc=True, width=40, depth=2, use_lr_decay=True, iters=WANG_ITERS)
ablation_results["A3 no LR decay"] = train_pi_deeponet("A3: no learning rate decay", use_bc=True, width=WANG_WIDTH, depth=WANG_DEPTH, use_lr_decay=False, iters=WANG_ITERS)
ablation_results["A4 short training"] = train_pi_deeponet("A4: short training (40k)", use_bc=True, width=WANG_WIDTH, depth=WANG_DEPTH, use_lr_decay=True, iters=40000)

print("\n" + "=" * 70)
print(f"  FINAL ABLATION RESULTS: {RUN_TAG}")
print("=" * 70)
print(f"{'variant':<30}{'rel L2 error':<24}")
print(f"{'PINN (Wang setup)':<30}{f'{pinn_error:.4%}':<24}")
print(f"{'DeepONet (Wang setup)':<30}{f'{don_mean:.4%} +/- {don_std:.4%}':<24}")
for name, (mean, std) in ablation_results.items():
    print(f"{'PI-DeepONet ' + name:<30}{f'{mean:.4%} +/- {std:.4%}':<24}")
print(f"{'Wang et al reported':<30}{'2.24% +/- 0.68%':<24}")
print("=" * 70)

rows = [
    ["PINN (Wang setup)", f"{pinn_error:.4%}", "one u(x)"],
    ["DeepONet (Wang setup)", f"{don_mean:.4%} +/- {don_std:.4%}", "operator, all Wang settings"],
]
for name, (mean, std) in ablation_results.items():
    rows.append([f"PI-DeepONet {name}", f"{mean:.4%} +/- {std:.4%}", "ablation" if "baseline" not in name else "all Wang settings"])
rows.append(["Wang et al (paper)", "2.24% +/- 0.68%", "PI-DeepONet reference"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
tbl = ax.table(cellText=rows, colLabels=["variant", "rel L2 error", "note"], loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.6)
ax.set_title("Advection ablation: which Wang hyperparameter matters most for PI-DeepONet?", fontsize=11, pad=10)
plt.savefig(f"{RUN_TAG}_metrics_table.png", dpi=200, bbox_inches="tight")
plt.show()

variants = ["Wang baseline"] + [k for k in ablation_results if k != "Wang baseline"]
means = [ablation_results[v][0] * 100 for v in variants]
stds = [ablation_results[v][1] * 100 for v in variants]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["tab:green"] + ["tab:red"] * 4
ax.bar(variants, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.axhline(2.24, color="black", ls="--", label="Wang et al reported (2.24%)")
ax.set_ylabel("rel L2 error (%)")
ax.set_title("PI-DeepONet ablation: bars higher than baseline = hyperparameter matters")
ax.legend()
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(f"{RUN_TAG}_bar_chart.png", dpi=200, bbox_inches="tight")
plt.show()

np.save("advection_ablation_results.npy", {"pinn": pinn_error, "don_mean": don_mean, "don_std": don_std, "ablations": ablation_results})